# Enrich `cleaned_data.csv` with Kaggle-derived household features

This notebook downloads or reads the Sri Lankan residential electricity consumption Kaggle dataset, derives the requested high-impact features, and saves a new CSV in `backend/data`.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional
import zipfile

import numpy as np
import pandas as pd

PROJECT_DIR = Path('/Users/rashinidissanayake/electricity-consumption-forcasting')
DATA_DIR = PROJECT_DIR / 'backend' / 'data'
CLEANED_PATH = DATA_DIR / 'data_clean.csv'
OUTPUT_PATH = DATA_DIR / 'cleaned_data_enriched.csv'
KAGGLE_DATASET_SLUG = 'lirneasia/sri-lankan-residential-electricity-consumption'
KAGGLE_ARCHIVE_PATH = Path.home() / '.cache' / 'kagglehub' / 'datasets' / 'lirneasia' / 'sri-lankan-residential-electricity-consumption' / '2.archive'

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print(f'Cleaned data path: {CLEANED_PATH}')
print(f'Output path: {OUTPUT_PATH}')
print(f'Kaggle archive path: {KAGGLE_ARCHIVE_PATH}')

Project directory: /Users/rashinidissanayake/electricity-consumption-forcasting
Cleaned data path: /Users/rashinidissanayake/electricity-consumption-forcasting/backend/data/data_clean.csv
Output path: /Users/rashinidissanayake/electricity-consumption-forcasting/backend/data/cleaned_data_enriched.csv
Kaggle archive path: /Users/rashinidissanayake/.cache/kagglehub/datasets/lirneasia/sri-lankan-residential-electricity-consumption/2.archive


In [2]:
def _normalize_column_name(name: str) -> str:
    return str(name).strip().lower().replace(' ', '_')


def find_first_existing_column(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    columns = list(df.columns)
    normalized_columns = {_normalize_column_name(column): column for column in columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        normalized_candidate = _normalize_column_name(candidate)
        if normalized_candidate in normalized_columns:
            return normalized_columns[normalized_candidate]
    return None


def find_columns_matching(df: pd.DataFrame, patterns: Iterable[str]) -> list[str]:
    matched = []
    for column in df.columns:
        lowered = column.lower()
        if any(pattern in lowered for pattern in patterns):
            matched.append(column)
    return matched


def normalize_boolean_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float) > 0

    truthy = {'1', 'true', 't', 'yes', 'y', 'on', 'inverter', 'led', 'solar'}
    normalized = series.fillna('').astype(str).str.strip().str.lower()
    return normalized.isin(truthy)


def canonical_rename(df: pd.DataFrame) -> pd.DataFrame:
    renamed = df.copy()
    renamed.columns = [column.strip() for column in renamed.columns]
    return renamed


def read_csv_from_archive(filename: str) -> pd.DataFrame:
    if not KAGGLE_ARCHIVE_PATH.exists():
        raise FileNotFoundError(f'Kaggle archive not found at {KAGGLE_ARCHIVE_PATH}')

    with zipfile.ZipFile(KAGGLE_ARCHIVE_PATH) as archive:
        members = [member for member in archive.namelist() if member.endswith(f'/{filename}') or member == filename]
        if not members:
            raise FileNotFoundError(f'Could not find {filename} inside {KAGGLE_ARCHIVE_PATH}')
        with archive.open(members[0]) as handle:
            return canonical_rename(pd.read_csv(handle))


def read_csv_from_search(root: Path, filename: str) -> pd.DataFrame:
    matches = list(root.rglob(filename))
    if matches:
        return canonical_rename(pd.read_csv(matches[0]))

    return read_csv_from_archive(filename)


def get_dataset_root() -> Path:
    return DATA_DIR


def get_household_id_column(df: pd.DataFrame) -> str:
    candidate = find_first_existing_column(df, ['household_id', 'household_ID', 'householdid', 'hhid', 'household'])
    if candidate is None:
        raise ValueError(f'Could not determine household id column from: {list(df.columns)}')
    return candidate


def get_month_column(df: pd.DataFrame) -> Optional[str]:
    return find_first_existing_column(df, ['month', 'billing_month', 'survey_month'])

In [3]:
def build_inverter_ac_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w1_ac_roster.csv')
    household_col = get_household_id_column(df)
    flag_col = find_first_existing_column(
        df,
        [
            'is_the_ac_inverter_or_not',
            'inverter_flag',
            'inverter',
            'is_inverter',
            'has_inverter',
            'ac_inverter',
            'inverter_ac',
        ],
    )

    if flag_col is None:
        text_columns = [column for column in df.columns if df[column].dtype == object]
        inverter_mask = pd.Series(False, index=df.index)
        for column in text_columns:
            inverter_mask |= df[column].fillna('').astype(str).str.contains('inverter', case=False, na=False)
    else:
        inverter_mask = normalize_boolean_series(df[flag_col])

    summary = df.assign(_is_inverter=inverter_mask).groupby(household_col, as_index=False).agg(
        inverter_ac=('_is_inverter', 'sum'),
        total_ac=('_is_inverter', 'size'),
    )
    summary['inverter_ac'] = summary['inverter_ac'].astype(int)
    summary['non_inverter_ac'] = (summary['total_ac'] - summary['inverter_ac']).clip(lower=0).astype(int)
    return summary[[household_col, 'inverter_ac', 'non_inverter_ac']].rename(columns={household_col: 'household_id'})


def build_solar_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w1_electricity_generation_water_heating_cooking.csv')
    household_col = get_household_id_column(df)

    solar_col = find_first_existing_column(df, ['generate_electicity_using_solar_energy'])
    water_heater_col = find_first_existing_column(df, ['solar_energy_used_for_water_heating'])

    if solar_col is None:
        solar_related = find_columns_matching(df, ['solar'])
        direct_solar = [column for column in solar_related if 'water' not in column.lower() and 'heater' not in column.lower() and 'geyser' not in column.lower()]
        solar_mask = pd.Series(False, index=df.index)
        for column in direct_solar:
            solar_mask |= normalize_boolean_series(df[column])
    else:
        solar_mask = normalize_boolean_series(df[solar_col])

    if water_heater_col is None:
        solar_related = find_columns_matching(df, ['solar'])
        water_related = [column for column in solar_related if any(token in column.lower() for token in ['water', 'heater', 'geyser'])]
        water_mask = pd.Series(False, index=df.index)
        for column in water_related:
            water_mask |= normalize_boolean_series(df[column])
    else:
        water_mask = normalize_boolean_series(df[water_heater_col])

    summary = df.assign(_has_solar=solar_mask, _water_heater_solar=water_mask).groupby(household_col, as_index=False).agg(
        has_solar=('_has_solar', 'max'),
        water_heater_solar=('_water_heater_solar', 'max'),
    )
    summary['has_solar'] = summary['has_solar'].astype(int)
    summary['water_heater_solar'] = summary['water_heater_solar'].astype(int)
    return summary.rename(columns={household_col: 'household_id'})


def build_room_count_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w1_room_roster.csv')
    household_col = get_household_id_column(df)
    room_identifier = find_first_existing_column(df, ['room_id', 'room_ID', 'room_no', 'room_number', 'room'])
    if room_identifier is None:
        summary = df.groupby(household_col, as_index=False).size().rename(columns={'size': 'room_count'})
    else:
        summary = df.groupby(household_col, as_index=False)[room_identifier].nunique().rename(columns={room_identifier: 'room_count'})
    return summary.rename(columns={household_col: 'household_id'})


def build_led_ratio_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w1_light_roster.csv')
    household_col = get_household_id_column(df)
    type_column = find_first_existing_column(
        df,
        [
            'type_of_the_bulb',
            'bulb_type',
            'light_type',
            'lamp_type',
            'type',
            'technology',
            'bulb',
        ],
    )

    if type_column is None:
        text_columns = [column for column in df.columns if df[column].dtype == object]
        if not text_columns:
            raise ValueError('Could not infer a light type column from w1_light_roster.csv')
        type_column = text_columns[0]

    normalized_type = df[type_column].fillna('').astype(str).str.strip().str.lower()
    led_mask = normalized_type.str.contains('led', na=False)
    summary = df.assign(_is_led=led_mask).groupby(household_col, as_index=False).agg(
        led_bulbs=('_is_led', 'sum'),
        total_bulbs=('_is_led', 'size'),
    )
    summary['led_bulbs'] = summary['led_bulbs'].astype(float)
    summary['total_bulbs'] = summary['total_bulbs'].astype(float)
    summary['led_ratio'] = np.where(summary['total_bulbs'] > 0, summary['led_bulbs'] / summary['total_bulbs'], 0.0)
    return summary[[household_col, 'led_ratio']].rename(columns={household_col: 'household_id'})


def build_tou_awareness_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w3_awareness_of_time_of_use.csv')
    household_col = get_household_id_column(df)
    awareness_col = find_first_existing_column(
        df,
        [
            'have_heard_about_time_of_use_metering',
            'tou_aware',
            'time_of_use',
            'awareness',
            'aware',
            'knowledge_of_time_of_use',
        ],
    )

    if awareness_col is None:
        awareness_col = [column for column in df.columns if column != household_col][0]

    summary = df.assign(_tou_aware=normalize_boolean_series(df[awareness_col])).groupby(household_col, as_index=False).agg(
        tou_aware=('_tou_aware', 'max')
    )
    summary['tou_aware'] = summary['tou_aware'].astype(int)
    return summary.rename(columns={household_col: 'household_id'})


def build_ac_hours_features(dataset_root: Path) -> pd.DataFrame:
    df = read_csv_from_search(dataset_root, 'w2_ac_roster.csv')
    household_col = get_household_id_column(df)
    day_hours_col = find_first_existing_column(
        df,
        [
            'no_of_hours_bulbs_was_on_during_daytime_last_week',
            'no_of_hours_ac_was_on_during_daytime_last_week',
            'daytime_hours',
            'hours_daytime',
        ],
    )
    night_hours_col = find_first_existing_column(
        df,
        [
            'no_of_hours_bulbs_was_on_during_night_last_week',
            'no_of_hours_ac_was_on_during_night_last_week',
            'nighttime_hours',
            'hours_nighttime',
        ],
    )

    if day_hours_col is None or night_hours_col is None:
        raise ValueError('Could not infer AC runtime columns from w2_ac_roster.csv')

    summary = df[[household_col, day_hours_col, night_hours_col]].copy()
    summary[day_hours_col] = pd.to_numeric(summary[day_hours_col], errors='coerce').fillna(0)
    summary[night_hours_col] = pd.to_numeric(summary[night_hours_col], errors='coerce').fillna(0)
    summary['ac_hours_per_day'] = (summary[day_hours_col] + summary[night_hours_col]) / 7.0
    summary['ac_hours_per_day'] = summary['ac_hours_per_day'].clip(lower=0, upper=24)
    summary = summary.groupby(household_col, as_index=False)['ac_hours_per_day'].mean()
    return summary.rename(columns={household_col: 'household_id'})


def build_load_variance_features(dataset_root: Path) -> pd.DataFrame:
    smart_files = [f'smart_6hour_{index}.csv' for index in range(1, 6)]
    monthly_frames = []

    for file_name in smart_files:
        df = read_csv_from_search(dataset_root, file_name)
        household_col = get_household_id_column(df)
        date_col = find_first_existing_column(df, ['DATE'])
        value_col = find_first_existing_column(df, ['TOTAL_IMPORT (kWh)', 'TOTAL_IMPORT'])
        if date_col is None or value_col is None:
            raise ValueError(f'Could not find DATE and TOTAL_IMPORT columns in {file_name}')

        df = df[[household_col, date_col, value_col]].copy()
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
        df = df.dropna(subset=[date_col, value_col])
        df['month'] = df[date_col].dt.month

        monthly = df.groupby([household_col, 'month'], as_index=False)[value_col].std(ddof=0).rename(columns={household_col: 'household_id', value_col: 'load_variance'})
        monthly_frames.append(monthly)

    combined = pd.concat(monthly_frames, ignore_index=True)
    combined = combined.groupby(['household_id', 'month'], as_index=False)['load_variance'].mean()
    return combined

In [8]:
dataset_root = get_dataset_root()
print(f'Using dataset root: {dataset_root}')

base = pd.read_csv(CLEANED_PATH)
base = canonical_rename(base)

feature_frames = [
    build_inverter_ac_features(dataset_root),
    build_solar_features(dataset_root),
    build_room_count_features(dataset_root),
    build_led_ratio_features(dataset_root),
    build_tou_awareness_features(dataset_root),
    build_ac_hours_features(dataset_root),
    build_load_variance_features(dataset_root),
]

merged = base.copy()
for frame in feature_frames:
    join_keys = ['household_id']
    if 'month' in frame.columns and 'month' in merged.columns:
        join_keys.append('month')
    merged = merged.merge(frame, on=join_keys, how='left')

fill_zero_columns = [
    'inverter_ac',
    'non_inverter_ac',
    'has_solar',
    'water_heater_solar',
    'room_count',
    'led_ratio',
    'tou_aware',
    'ac_hours_per_day',
    'load_variance',
]
for column in fill_zero_columns:
    if column in merged.columns:
        merged[column] = merged[column].fillna(0)

integer_columns = ['inverter_ac', 'non_inverter_ac', 'has_solar', 'water_heater_solar', 'room_count', 'tou_aware']
for column in integer_columns:
    if column in merged.columns:
        merged[column] = merged[column].round().astype(int)

merged.to_csv(OUTPUT_PATH, index=False)

print(f'Saved enriched dataset to: {OUTPUT_PATH}')
print(f'Original shape: {base.shape}')
print(f'Enriched shape: {merged.shape}')
merged.head()

Using dataset root: /Users/rashinidissanayake/electricity-consumption-forcasting/backend/data


/var/folders/t3/3qp73x1d4ps0kt17hk0r1kxm0000gn/T/ipykernel_220/2604746022.py:52: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  return canonical_rename(pd.read_csv(handle))
/var/folders/t3/3qp73x1d4ps0kt17hk0r1kxm0000gn/T/ipykernel_220/2604746022.py:52: DtypeWarning: Columns (9,27) have mixed types. Specify dtype option on import or set low_memory=False.
  return canonical_rename(pd.read_csv(handle))
/var/folders/t3/3qp73x1d4ps0kt17hk0r1kxm0000gn/T/ipykernel_220/2604746022.py:52: DtypeWarning: Columns (14,25,27) have mixed types. Specify dtype option on import or set low_memory=False.
  return canonical_rename(pd.read_csv(handle))
/var/folders/t3/3qp73x1d4ps0kt17hk0r1kxm0000gn/T/ipykernel_220/2604746022.py:52: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  return canonical_rename(pd.read_csv(handle))
/var/folders/t3/3qp73x1d4ps0kt17hk0r1kxm0000gn/T/ipykernel_220/2604746022.py:5

Saved enriched dataset to: /Users/rashinidissanayake/electricity-consumption-forcasting/backend/data/cleaned_data_enriched.csv
Original shape: (15105, 21)
Enriched shape: (15105, 30)


,household_id,month,monthly_kwh,prev1,prev2,prev3,target,peak_ratio,family_size,has_refrigerator,...,rain,inverter_ac,non_inverter_ac,has_solar,water_heater_solar,room_count,led_ratio,tou_aware,ac_hours_per_day,load_variance
0,ID0013,3,294.139,158.6889,140.9611,148.8060,175.646,0.076345,1.0,1.0,...,133.75,0,0,0,0,10,1.0,0,0.0,46.357412
1,ID0013,4,175.646,294.1390,158.6889,140.9611,149.361,0.162719,1.0,1.0,...,220.45,0,0,0,0,10,1.0,0,0.0,30.186854
2,ID0013,5,168.025,149.3610,175.6460,294.1390,149.978,0.244045,1.0,1.0,...,438.60,0,0,0,0,10,1.0,0,0.0,47.307423
3,ID0013,6,168.842,149.9780,168.0250,149.3610,164.528,0.226389,1.0,1.0,...,332.20,0,0,0,0,10,1.0,0,0.0,32.440552
4,ID0013,7,156.287,164.5280,168.8420,149.9780,159.222,0.166753,1.0,1.0,...,274.35,0,0,0,0,10,1.0,0,0.0,45.105325


## District + Month Range Inputs

This section prepares the next step of the project: accept a district and month range, fetch weather inputs when possible, and estimate peak ratio automatically.

In [5]:
from datetime import date
from calendar import monthrange
import requests

DISTRICT_COORDINATES = {
    'colombo': (6.9271, 79.8612),
    'gampaha': (7.0873, 80.0140),
    'kalutara': (6.5854, 79.9607),
    'kandy': (7.2906, 80.6337),
    'matale': (7.4675, 80.6234),
    'nuwara eliya': (6.9497, 80.7891),
    'galle': (6.0535, 80.2210),
    'matara': (5.9485, 80.5353),
    'hambantota': (6.1244, 81.1185),
    'jaffna': (9.6615, 80.0255),
    'kilinochchi': (9.3803, 80.3769),
    'mullaitivu': (9.2671, 80.8123),
    'vavuniya': (8.7514, 80.4971),
    'mannar': (8.9769, 79.9040),
    'baticaloa': (7.7384, 81.6987),
    'ampara': (7.2919, 81.6724),
    'trincomalee': (8.5874, 81.2152),
    'kurunegala': (7.4863, 80.3647),
    'puttalam': (8.0365, 79.8283),
    'anuradhapura': (8.3114, 80.4037),
    'polonnaruwa': (7.9391, 81.0006),
    'badulla': (6.9934, 81.0550),
    'monaragala': (6.8728, 81.3510),
    'ratnapura': (6.7056, 80.3848),
    'kegalle': (7.2525, 80.3436),
}


def resolve_district_coordinates(district: str) -> tuple[float, float]:
    key = str(district).strip().casefold()
    if key not in DISTRICT_COORDINATES:
        available = ', '.join(sorted(DISTRICT_COORDINATES))
        raise ValueError(f'Unknown district: {district}. Available districts: {available}')
    return DISTRICT_COORDINATES[key]


def expand_month_range(start_month: int, end_month: int, year: int | None = None) -> list[date]:
    if not 1 <= int(start_month) <= 12 or not 1 <= int(end_month) <= 12:
        raise ValueError('start_month and end_month must be between 1 and 12.')

    year = date.today().year if year is None else int(year)
    months = []
    current_month = int(start_month)
    current_year = year

    while True:
        months.append(date(current_year, current_month, 1))
        if current_month == int(end_month):
            break
        current_month += 1
        if current_month > 12:
            current_month = 1
            current_year += 1
    return months


def district_month_climatology() -> pd.DataFrame:
    district_column = find_first_existing_column(merged, ['district', 'district_name', 'districts'])

    if district_column is None:
        climatology = (
            merged.groupby(['month'], as_index=False)[['temp', 'humidity', 'rain', 'peak_ratio']]
            .mean()
            .rename(columns={
                'temp': 'climate_temp',
                'humidity': 'climate_humidity',
                'rain': 'climate_rain',
                'peak_ratio': 'climate_peak_ratio',
            })
        )
        climatology['district'] = 'all'
    else:
        climatology = (
            merged.groupby([district_column, 'month'], as_index=False)[['temp', 'humidity', 'rain', 'peak_ratio']]
            .mean()
            .rename(columns={
                district_column: 'district',
                'temp': 'climate_temp',
                'humidity': 'climate_humidity',
                'rain': 'climate_rain',
                'peak_ratio': 'climate_peak_ratio',
            })
        )

    climatology['district_key'] = climatology['district'].astype(str).str.strip().str.casefold()
    return climatology


CLIMATOLOGY_LOOKUP = district_month_climatology()
print('Prepared climatology lookup with rows:', len(CLIMATOLOGY_LOOKUP))

Prepared climatology lookup with rows: 11


In [6]:
OPEN_METEO_ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'


def fetch_monthly_weather_from_api(district: str, months: list[date]) -> pd.DataFrame:
    latest_month_start = date.today().replace(day=1)
    if months[-1] >= latest_month_start:
        return pd.DataFrame(columns=['district', 'month', 'temp', 'humidity', 'rain'])

    latitude, longitude = resolve_district_coordinates(district)
    start_date = months[0].strftime('%Y-%m-%d')
    last_day = monthrange(months[-1].year, months[-1].month)[1]
    end_date = date(months[-1].year, months[-1].month, last_day).strftime('%Y-%m-%d')

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum',
        'timezone': 'auto',
    }

    try:
        response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        daily = pd.DataFrame(payload.get('daily', {}))
        if daily.empty:
            return pd.DataFrame(columns=['district', 'month', 'temp', 'humidity', 'rain'])

        daily['date'] = pd.to_datetime(daily['time'])
        daily['month'] = daily['date'].dt.month
        monthly_api = daily.groupby('month', as_index=False).agg(
            temp=('temperature_2m_mean', 'mean'),
            humidity=('relative_humidity_2m_mean', 'mean'),
            rain=('precipitation_sum', 'sum'),
        )
        monthly_api['district'] = district
        return monthly_api[['district', 'month', 'temp', 'humidity', 'rain']]
    except Exception:
        return pd.DataFrame(columns=['district', 'month', 'temp', 'humidity', 'rain'])


def build_peak_ratio_estimator(history: pd.DataFrame):
    district_column = find_first_existing_column(history, ['district', 'district_name', 'districts'])
    global_peak = float(history['peak_ratio'].mean())

    if district_column is None:
        month_only = (
            history.groupby(['month'], as_index=False)['peak_ratio']
            .mean()
            .rename(columns={'peak_ratio': 'month_peak_ratio'})
        )

        def estimate(district: str, month: int) -> float:
            month_match = month_only[month_only['month'] == month]
            if not month_match.empty:
                return float(month_match['month_peak_ratio'].iloc[0])
            return global_peak

        return estimate

    district_month = (
        history.groupby([district_column, 'month'], as_index=False)['peak_ratio']
        .mean()
        .rename(columns={district_column: 'district', 'peak_ratio': 'district_month_peak_ratio'})
    )
    district_only = (
        history.groupby([district_column], as_index=False)['peak_ratio']
        .mean()
        .rename(columns={district_column: 'district', 'peak_ratio': 'district_peak_ratio'})
    )

    district_month['district_key'] = district_month['district'].astype(str).str.strip().str.casefold()
    district_only['district_key'] = district_only['district'].astype(str).str.strip().str.casefold()

    def estimate(district: str, month: int) -> float:
        district_key = str(district).strip().casefold()
        month_match = district_month[(district_month['district_key'] == district_key) & (district_month['month'] == month)]
        if not month_match.empty:
            return float(month_match['district_month_peak_ratio'].iloc[0])
        district_match = district_only[district_only['district_key'] == district_key]
        if not district_match.empty:
            return float(district_match['district_peak_ratio'].iloc[0])
        return global_peak

    return estimate


PEAK_RATIO_ESTIMATOR = build_peak_ratio_estimator(base)
print('Peak ratio estimator is ready.')

Peak ratio estimator is ready.


In [7]:
def build_prediction_inputs(district: str, start_month: int, end_month: int, year: int | None = None) -> pd.DataFrame:
    months = expand_month_range(start_month, end_month, year=year)
    month_numbers = [value.month for value in months]
    weather = fetch_monthly_weather_from_api(district, months)

    district_key = str(district).strip().casefold()
    climate = CLIMATOLOGY_LOOKUP[CLIMATOLOGY_LOOKUP['district_key'] == district_key]
    if climate.empty:
        climate = CLIMATOLOGY_LOOKUP[CLIMATOLOGY_LOOKUP['district_key'] == 'all']

    rows = []
    for month_number in month_numbers:
        api_row = weather[weather['month'] == month_number]
        if not api_row.empty:
            temp = float(api_row['temp'].iloc[0])
            humidity = float(api_row['humidity'].iloc[0])
            rain = float(api_row['rain'].iloc[0])
        else:
            fallback = climate[climate['month'] == month_number]
            if not fallback.empty:
                temp = float(fallback['climate_temp'].iloc[0])
                humidity = float(fallback['climate_humidity'].iloc[0])
                rain = float(fallback['climate_rain'].iloc[0])
            else:
                temp = float(merged['temp'].mean())
                humidity = float(merged['humidity'].mean())
                rain = float(merged['rain'].mean())

        peak_ratio = float(PEAK_RATIO_ESTIMATOR(district, month_number))
        rows.append(
            {
                'district': district,
                'month': month_number,
                'temp': round(temp, 3),
                'humidity': round(humidity, 3),
                'rain': round(rain, 3),
                'peak_ratio': round(peak_ratio, 6),
            }
        )

    return pd.DataFrame(rows)


sample_inputs = build_prediction_inputs('Colombo', 3, 5)
sample_inputs

,district,month,temp,humidity,rain,peak_ratio
0,Colombo,3,27.090,81.684,153.782,0.121642
1,Colombo,4,27.445,83.022,202.656,0.222863
2,Colombo,5,27.269,85.850,418.781,0.264158
